# 1. Run MedGemma 27B extraction

Start here for a normal research run on **Windows or Linux**. Ollama serves only **unquantized F16/BF16 MedGemma 27B**; Python verifies quoted evidence and applies the SCOGS tables.

**Before Run All:** follow [Ollama setup](../docs/ollama_setup.md), install `requirements.txt`, and select this project's Python kernel. Start Jupyter from the repository or `notebooks/` directory. No Colab, Drive, CUDA Python packages, or shell magics are needed.

This is a research harness, not a clinical decision tool. Quoted words appearing in a note do **not** prove that they support the extracted value.

## 1. Locate the project
Paths are resolved from the repository, not from a machine-specific folder. Subprocesses use the notebook's Python interpreter.

In [ ]:
import json
import os
import pathlib
import shutil
import subprocess
import sys
from datetime import datetime, timezone

ROOT = next((p for p in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents)
             if (p / 'scripts/experiments/medgemma_extraction.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Start Jupyter from the cloned st_jude repository.')
sys.path.insert(0, str(ROOT / 'scripts'))
from experiments.review_results import export_reviews, summarize

SCRIPT = ROOT / 'scripts/experiments/medgemma_extraction.py'
print(f'Project: {ROOT}\nPython: {sys.executable}')

## 2. Configure this run
Edit only this cell for most runs. Start with `NOTES = 2` to check your hardware, then use 20. Stage `3` is the current candidate prompt, not a claim of validated accuracy. `scd_primary` selects disease-focused cases; `loose` also includes mention-only cases for absence audits.

Keep `CONCURRENCY = 1` for repeatability comparisons. A full model needs roughly **54 GB just for weights**, plus context/runtime memory. Increase `TIMEOUT` for CPU offload and `NUM_CTX` for long notes; both may increase run cost.

In [ ]:
MODEL = 'medgemma-27b-f16'  # Your local F16 tag; a verified BF16 tag also works.
HOST = os.environ.get('OLLAMA_HOST', 'http://localhost:11434')
NOTES = 20
# 14 focus outcomes: Pain, Stroke, SS, ACS, Priapism, Chronic Pain, CKD,
# Retinopathy, CD, Depression, TCD Elevation, Asthma, AVN, Leg Ulcer (or 'all').
OUTCOMES = '10,11,12,15,17,21,24,28,29,39,40,47,48,49'
COHORT = 'scd_primary'
PROMPT_STAGE = '2b'  # default; any of '0', '1', '2a', '2b', '3'
REPEAT = 2
CONCURRENCY = 1
NUM_CTX = 16384
TIMEOUT = 600
RUN_DIR = ROOT / 'results' / datetime.now(timezone.utc).strftime('run_%Y%m%dT%H%M%S_%fZ')

## 3. Verify the model before processing notes
This checks Ollama's actual parameter count, architecture, precision, model digest, and a synthetic JSON response. A misleading tag name is not enough to pass. Any error stops the notebook. It does not download or replace your model.

In [ ]:
base_command = [sys.executable, str(SCRIPT), '--model', MODEL, '--host', HOST,
                '--timeout', str(TIMEOUT), '--num-ctx', str(NUM_CTX)]
subprocess.run(base_command + ['--check-model'], cwd=ROOT, check=True)

## 4. Extract features and grade them
The bundled case cache is enough; no dataset download is required. The same CLI powers this notebook and terminal runs. Every run gets a new folder, and existing result files are never overwritten. Re-running this cell requires a new `RUN_DIR`.

In [ ]:
OUT_PATH = RUN_DIR / f'stage_{PROMPT_STAGE}.json'
command = base_command + [
    '--notes', str(NOTES), '--outcomes', OUTCOMES, '--cohort', COHORT,
    '--stratify', '--holdout-frac', '0.25', '--repeat', str(REPEAT),
    '--concurrency', str(CONCURRENCY), '--prompt-stage', PROMPT_STAGE,
    '--out', str(OUT_PATH),
]
subprocess.run(command, cwd=ROOT, check=True)
data = json.loads(OUT_PATH.read_text(encoding='utf-8'))
print(f'Saved: {OUT_PATH}')

## 5. Inspect the results
Read per-outcome counts, not just pooled percentages. `absent` means the model did not call the outcome present; `refuted` means the model called it present but the rules found no matching grade. `grade_set` and `cannot_grade` preserve missing or ambiguous evidence.

Quote verification measures **grounding**, not precision. Temperature-zero consistency is a serving check, not a measure of clinical accuracy.

In [ ]:
print(json.dumps(summarize(data), indent=2, ensure_ascii=False))

## 6. Export human-review worksheets
The exporter reads `accepted_findings` and `conflicts` directly from this run, rather than trying to reconstruct them from scalar feature values. All sheets carry a content-derived `run_id`.

- `handcheck.csv`: up to 100 grounded findings. Mark `supports_value` as `y` or `n`; precision is `y / (y + n)`.
- `absence_audit.csv`: up to 50 absent pairs, including note text. Mark `truly_absent`; the sampled false-negative fraction is `n / (y + n)`.
- `conflicts.csv` and `refuted_audit.csv`: inspect every withheld conflict and rule-refuted presence call.

Empty sheets still have headers. Existing sheets are never overwritten. Read five full notes separately to look for missed findings; these automated counters cannot measure recall.

In [ ]:
review_paths = export_reviews(OUT_PATH)
for name, path in review_paths.items():
    print(f'{name}: {path}')

record = data['detailed_records'][0]
print('\nFirst case:', record['patient_uid'], record['title'])
print(record['patient_note'])
for number, outcome in record['outcomes'].items():
    print(number, outcome['outcome_name'])
    print(json.dumps(outcome['accepted_findings'], indent=2, ensure_ascii=False))

## 7. Keep the artifacts together
The folder already contains the JSON results and review sheets. This creates a ZIP beside it for sharing with authorized collaborators. If you edit the CSVs later, rerun this cell to update the ZIP. Do not commit patient-level exports or notebook outputs.

In [ ]:
archive = shutil.make_archive(str(RUN_DIR), 'zip', root_dir=RUN_DIR)
print(f'Artifacts: {RUN_DIR}\nArchive: {archive}')